# 01 · Extraction et ingestion bronze

**Objectif** : copier les instantanés Vélib collectés (`data/velib_status/*.csv`) dans le data lake
(RustFS) au format **Delta**, **sans aucune transformation** : c'est la couche *bronze*.

- Entrée : fichiers CSV bruts, un par instantané (créés par `src/collecter_velib.py`).
- Sortie : table Delta `s3a://velib/bronze/velib_status`, partitionnée par date d'instantané.
- Seuls ajouts, pour pouvoir remonter à la source : `_source_file`, `_snapshot_ts`, `_snapshot_date`.
- Toutes les colonnes d'origine restent en **texte** : le typage et le nettoyage se font en silver (notebook 02).
- Seuls les **noms** de colonnes sont normalisés (sans espaces ni accents), car Delta les refuse.

**Prérequis** : `docker compose up -d` (RustFS démarré), au moins un instantané dans `data/velib_status/`,
noyau Python = `.venv`.

In [ ]:
import sys
from pathlib import Path

# Racine du projet : on remonte depuis le dossier courant jusqu'à trouver docker-compose.yml
# (fonctionne que le notebook soit lancé depuis notebooks/ ou depuis la racine)
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.insert(0, str(RACINE / "src"))
print("Racine du projet :", RACINE)

import config
from lake import creer_bucket
from spark_session import get_spark


Racine du projet : d:\Users\HP\Downloads\projet-dataviz-SupDeVince\projet-dataviz


## 1. Le bucket du data lake

In [ ]:
creer_bucket()

Bucket 'velib' déjà présent


## 2. Les instantanés à ingérer

Le séparateur du CSV n'est pas garanti (`;` ou `,`) : on le détecte sur la première ligne.

In [ ]:
DATA = Path(config.DATA_DIR)
DATA = DATA if DATA.is_absolute() else RACINE / DATA
DOSSIER = DATA / "velib_status"
fichiers = sorted(DOSSIER.glob("velib_*.csv"))
print(f"{len(fichiers)} instantanés dans {DOSSIER}")
assert fichiers, "Aucun instantané : lancer la collecte (README, étape 2)"
print("Premier :", fichiers[0].name)
print("Dernier :", fichiers[-1].name)

entete = fichiers[0].read_text(encoding="utf-8-sig").splitlines()[0]
SEP = max([";", ",", "\t"], key=entete.count)
print("Séparateur détecté :", repr(SEP))
print("Colonnes :", entete.split(SEP))

596 instantanés dans D:\Users\HP\Downloads\projet-dataviz-SupDeVince\projet-dataviz\data\velib_status
Premier : velib_20261109T155809Z.csv
Dernier : velib_20261110T122644Z.csv
Séparateur détecté : ';'
Colonnes : ['stationcode', 'name', 'is_installed', 'capacity', 'numdocksavailable', 'numbikesavailable', 'mechanical', 'ebike', 'is_renting', 'is_returning', 'duedate', 'coordonnees_geo', 'nom_arrondissement_communes', 'code_insee_commune', 'station_opening_hours']


## 3. Lecture brute et écriture en Delta

Aucun typage (`inferSchema` désactivé : tout est en texte), aucune ligne retirée.

In [ ]:
import re
import unicodedata

from pyspark.sql import functions as F

spark = get_spark("01_extraction_bronze")
spark.conf.set("spark.sql.session.timeZone", "UTC")  # les noms de fichiers sont en UTC

brut = (
    spark.read
    .option("header", True)
    .option("sep", SEP)
    .option("encoding", "UTF-8")
    .csv(DOSSIER.as_uri() + "/velib_*.csv")
)

# Delta refuse les espaces, accents et caractères spéciaux dans les noms de colonnes :
# on les normalise (seul changement apporté aux colonnes d'origine, leurs valeurs restent intactes).
def normaliser(nom: str) -> str:
    sans_accents = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", sans_accents).strip("_").lower()


colonnes = {c: normaliser(c) for c in brut.columns}
assert len(set(colonnes.values())) == len(colonnes), f"deux colonnes ont le même nom normalisé : {colonnes}"
brut = brut.toDF(*colonnes.values())
print("Colonnes renommées :")
for ancien, nouveau in colonnes.items():
    print(f"  {ancien!r:45} -> {nouveau}")

bronze = (
    brut
    .withColumn("_source_file", F.input_file_name())
    .withColumn(
        "_snapshot_ts",
        F.to_timestamp(F.regexp_extract("_source_file", r"velib_(\d{8}T\d{6})Z", 1), "yyyyMMdd'T'HHmmss"),
    )
    .withColumn("_snapshot_date", F.to_date("_snapshot_ts"))
)

BRONZE = f"{config.BRONZE_PATH}/velib_status"
(
    bronze.write.format("delta")
    .mode("overwrite")              # on peut relancer le notebook : le résultat est identique
    .option("overwriteSchema", True)
    .partitionBy("_snapshot_date")
    .save(BRONZE)
)
print("Écrit dans", BRONZE)

Colonnes renommées :
  'Identifiant station'                         -> identifiant_station
  'Nom station'                                 -> nom_station
  'Station en fonctionnement'                   -> station_en_fonctionnement
  'Capacité de la station'                      -> capacite_de_la_station
  'Nombre bornettes libres'                     -> nombre_bornettes_libres
  'Nombre total vélos disponibles'              -> nombre_total_velos_disponibles
  'Vélos mécaniques disponibles'                -> velos_mecaniques_disponibles
  'Vélos électriques disponibles'               -> velos_electriques_disponibles
  'Borne de paiement disponible'                -> borne_de_paiement_disponible
  'Retour vélib possible'                       -> retour_velib_possible
  'Actualisation de la donnée'                  -> actualisation_de_la_donnee
  'Coordonnées géographiques'                   -> coordonnees_geographiques
  'Nom communes équipées'                       -> nom_communes_equi

## 4. Vérification

Le nombre de lignes lues dans Delta doit être égal au nombre de lignes des CSV : rien n'a été perdu ni modifié.

In [ ]:
lu = spark.read.format("delta").load(BRONZE)
n_bronze = lu.count()
n_csv = brut.count()
print(f"Lignes CSV : {n_csv:,} | lignes bronze : {n_bronze:,}".replace(",", " "))
assert n_bronze == n_csv, "écart entre les CSV et la table bronze"
print(f"≈ {100 * n_bronze / 2_000_000:.0f} % de l'objectif de 2 millions de lignes")

lu.groupBy("_snapshot_date").count().orderBy("_snapshot_date").show(50)
lu.printSchema()
lu.show(5, truncate=False)

Lignes CSV : 907 101 | lignes bronze : 907 101
≈ 45 % de l'objectif de 2 millions de lignes
+--------------+------+
|_snapshot_date| count|
+--------------+------+
|    2026-11-09|366799|
|    2026-11-10|540302|
+--------------+------+

root
 |-- identifiant_station: string (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- station_en_fonctionnement: string (nullable = true)
 |-- capacite_de_la_station: string (nullable = true)
 |-- nombre_bornettes_libres: string (nullable = true)
 |-- nombre_total_velos_disponibles: string (nullable = true)
 |-- velos_mecaniques_disponibles: string (nullable = true)
 |-- velos_electriques_disponibles: string (nullable = true)
 |-- borne_de_paiement_disponible: string (nullable = true)
 |-- retour_velib_possible: string (nullable = true)
 |-- actualisation_de_la_donnee: string (nullable = true)
 |-- coordonnees_geographiques: string (nullable = true)
 |-- nom_communes_equipees: string (nullable = true)
 |-- code_insee_communes_equipees:

## À retenir pour le README

- Nombre d'instantanés et de lignes en bronze : *à noter ici après l'exécution*
- Période couverte : *à noter ici*
- Les colonnes sont toutes en texte, le typage est fait en silver.